# Buscar inconsistencias en los archivos CSV para crear tablas solidas y que filtren correctamente datos defectuosos

## Archivo orders.csv

In [18]:
import pandas as pd

print("--- FASE 1: AUDITORÍA DE DATOS CRUDOS ---")

#cargar el nucleo de transaccion
df_orders = pd.read_csv('data/orders.csv')

#analizar dimensiones
print(f"Total de pedidos registrados: {len(df_orders)}")

#analizar tipos de datos de las columnas
print(f'1. Los tipos de datos de las columnas son:\n{df_orders.dtypes}\n')

print("\n2. Contar valores nulos de las columnas")
print(df_orders.isnull().sum())

--- FASE 1: AUDITORÍA DE DATOS CRUDOS ---
Total de pedidos registrados: 120000
1. Los tipos de datos de las columnas son:
order_id            int64
customer_id         int64
order_datetime        str
channel               str
currency              str
current_status        str
is_active           int64
deleted_at            str
order_total       float64
dtype: object


2. Contar valores nulos de las columnas
order_id               0
customer_id            0
order_datetime         0
channel                0
currency               0
current_status         0
is_active              0
deleted_at        119395
order_total            0
dtype: int64


In [19]:
print("--- ANÁLISIS DE DOMINIOS CATEGÓRICOS ---")

#extraer valores unicos para definir los limites en las tablas
estados_unicos = df_orders['current_status'].unique()
canales_unicos = df_orders['channel'].unique()
monedas_unicas = df_orders['currency'].unique()

print(f"Estados posibles detectados ({len(estados_unicos)}): {estados_unicos}\n")
print(f"Canales de venta detectados ({len(canales_unicos)}): {canales_unicos}\n")
print(f"Monedas operativas detectadas ({len(monedas_unicas)}): {monedas_unicas}")

--- ANÁLISIS DE DOMINIOS CATEGÓRICOS ---
Estados posibles detectados (7): <StringArray>
['shipped', 'created', 'delivered', 'paid', 'cancelled', 'packed', 'refunded']
Length: 7, dtype: str

Canales de venta detectados (4): <StringArray>
['mobile', 'web', 'store', 'phone']
Length: 4, dtype: str

Monedas operativas detectadas (2): <StringArray>
['PYG', 'USD']
Length: 2, dtype: str


In [20]:
print("--- AUDITORÍA DE LÓGICA DE NEGOCIO ---")

#un pedido no puede valer 0 o menos 
pedidos_invalidos = df_orders[df_orders['order_total'] <= 0]

print(f"ALERTA: Se detectaron {len(pedidos_invalidos)} pedidos con monto cero o negativo.")
print("Conclusión: Es obligatorio implementar CHECK (order_total > 0) en el esquema SQL.")

#mostrar el fraude jeje
display(pedidos_invalidos[['order_id', 'customer_id', 'current_status', 'order_total']].head())

--- AUDITORÍA DE LÓGICA DE NEGOCIO ---
ALERTA: Se detectaron 5934 pedidos con monto cero o negativo.
Conclusión: Es obligatorio implementar CHECK (order_total > 0) en el esquema SQL.


,order_id,customer_id,current_status,order_total
13,14,16720,paid,0.0
18,19,26983,shipped,0.0
25,26,26418,delivered,0.0
61,62,15719,paid,0.0
67,68,24631,packed,0.0


## Archivo customers.csv

In [24]:
print("=== PERFILADO DE CUSTOMERS.CSV ===")
df_customers = pd.read_csv('data/customers.csv')

print("\n1. Tipos de datos por fila:\n")
print(df_customers.dtypes)

print("\n2. Valores nulos por fila:\n")
print(df_customers.isnull().sum())

=== PERFILADO DE CUSTOMERS.CSV ===

1. Tipos de datos por fila:

customer_id    int64
full_name        str
email            str
phone          int64
city             str
segment          str
created_at       str
is_active      int64
deleted_at       str
dtype: object

2. Valores nulos por fila:

customer_id        0
full_name          0
email              0
phone              0
city               0
segment            0
created_at         0
is_active          0
deleted_at     29535
dtype: int64


In [35]:
print("--- ANÁLISIS DE DOMINIOS CATEGÓRICOS ---")

#extraer valores unicos importantes para el check
segmentos_unicos = df_customers['segment'].unique()
estados_actividad = df_customers['is_active'].unique()

print(f"-Segmentos comerciales detectados: {segmentos_unicos}\n")
print(f"-Valores de actividad (is_active): {estados_actividad}")
print("\nConclusión: Se requieren reglas CHECK para limitar 'segment' a estos valores y 'is_active' estrictamente a 0 o 1.")

--- ANÁLISIS DE DOMINIOS CATEGÓRICOS ---
-Segmentos comerciales detectados: <StringArray>
['retail', 'wholesale', 'online_only', 'vip']
Length: 4, dtype: str

-Valores de actividad (is_active): [0 1]

Conclusión: Se requieren reglas CHECK para limitar 'segment' a estos valores y 'is_active' estrictamente a 0 o 1.


In [39]:
print("--- AUDITORÍA DE INTEGRIDAD: TELÉFONOS Y EMAILS ---")

#justificacion del tipo de dato del telefono
print("Muestra de formato de teléfonos:")
display(df_customers['phone'].head(3))
print("Diagnóstico: El teléfono incluye el prefijo internacional (+595). Si la base de datos intenta guardarlo como INTEGER, el motor fallará \npor error de sintaxis o destruirá el signo '+'. Obligatorio usar TEXT.\n")

#justificación de la regla UNIQUE para correos
correos_duplicados = df_customers.duplicated(subset=['email']).sum()
print(f"Correos duplicados encontrados en el CSV: {correos_duplicados}")
print("Diagnóstico: Como no hay duplicados, es seguro y obligatorio blindar la columna email con la restricción UNIQUE en SQL \npara evitar registros dobles en el futuro.")

--- AUDITORÍA DE INTEGRIDAD: TELÉFONOS Y EMAILS ---
Muestra de formato de teléfonos:


0    595992795509
1    595930861340
2    595932679156
Name: phone, dtype: int64

Diagnóstico: El teléfono incluye el prefijo internacional (+595). Si la base de datos intenta guardarlo como INTEGER, el motor fallará 
por error de sintaxis o destruirá el signo '+'. Obligatorio usar TEXT.

Correos duplicados encontrados en el CSV: 0
Diagnóstico: Como no hay duplicados, es seguro y obligatorio blindar la columna email con la restricción UNIQUE en SQL 
para evitar registros dobles en el futuro.


## Archivo products.csv

In [40]:
print("=== PERFILADO DE PRODUCTS.CSV ===")
df_products = pd.read_csv('data/products.csv')

print("\n1. Tipos de datos por fila:\n")
print(df_products.dtypes)

print("\n2. Valores nulos por fila:\n")
print(df_products.isnull().sum())

=== PERFILADO DE PRODUCTS.CSV ===

1. Tipos de datos por fila:

product_id        int64
sku                 str
product_name        str
category            str
brand               str
unit_price      float64
unit_cost       float64
created_at          str
is_active         int64
deleted_at          str
dtype: object

2. Valores nulos por fila:

product_id         0
sku                0
product_name       0
category           0
brand              0
unit_price         0
unit_cost          0
created_at         0
is_active          0
deleted_at      7952
dtype: int64


In [41]:
print("--- ANÁLISIS DE UNICIDAD Y DOMINIOS ---")

#verificar codigo de productos duplicados
skus_duplicados = df_products.duplicated(subset=['sku']).sum()
print(f"SKUs duplicados encontrados: {skus_duplicados}")
print(f"Diagnóstico: Aunque existan {skus_duplicados} duplicados actuales, la regla UNIQUE en la columna 'sku' es innegociable para prevenir \ncolisiones futuras.\n")

#limite de actividad de 0 a 1 
estados_actividad = df_products['is_active'].unique()
print(f"Valores de actividad (is_active): {estados_actividad}")
print("Conclusión: Mantener CHECK (is_active IN (0, 1)).")

--- ANÁLISIS DE UNICIDAD Y DOMINIOS ---
SKUs duplicados encontrados: 0
Diagnóstico: Aunque existan 0 duplicados actuales, la regla UNIQUE en la columna 'sku' es innegociable para prevenir 
colisiones futuras.

Valores de actividad (is_active): [1 0]
Conclusión: Mantener CHECK (is_active IN (0, 1)).


In [42]:
print("--- AUDITORÍA DE INTEGRIDAD FINANCIERA ---")

#el precio no puede ser 0 o menor
precios_cero = df_products[(df_products['unit_price'] <= 0) | (df_products['unit_cost'] <= 0)]
print(f"Alerta de precios/costos <= 0: {len(precios_cero)} registros encontrados.")

#no se puede vender mas barato de lo que se consiguio 
productos_a_perdida = df_products[df_products['unit_cost'] >= df_products['unit_price']]
print(f"Anomalía: Se detectaron {len(productos_a_perdida)} productos configurados para venderse a pérdida o a precio de costo neto.")

print("\nDiagnóstico Final:")
print("Es estrictamente obligatorio levantar tres muros lógicos (CHECK) en SQL:")
print("1. CHECK (unit_price > 0)")
print("2. CHECK (unit_cost > 0)")
print("3. CHECK (unit_price > unit_cost) -> Validación cruzada de rentabilidad.")

--- AUDITORÍA DE INTEGRIDAD FINANCIERA ---
Alerta de precios/costos <= 0: 0 registros encontrados.
Anomalía: Se detectaron 0 productos configurados para venderse a pérdida o a precio de costo neto.

Diagnóstico Final:
Es estrictamente obligatorio levantar tres muros lógicos (CHECK) en SQL:
1. CHECK (unit_price > 0)
2. CHECK (unit_cost > 0)
3. CHECK (unit_price > unit_cost) -> Validación cruzada de rentabilidad.


## Archivo payments.csv

In [43]:
print("=== PERFILADO DE PAYMENTS.CSV ===")
df_payments = pd.read_csv('data/payments.csv')
print("\n1. Tipos de datos por fila:\n")
print(df_payments.dtypes)

print("\n2. Valores nulos por fila:\n")
print(df_payments.isnull().sum())


=== PERFILADO DE PAYMENTS.CSV ===

1. Tipos de datos por fila:

payment_id            int64
order_id              int64
payment_datetime        str
method                  str
payment_status          str
amount              float64
currency                str
dtype: object

2. Valores nulos por fila:

payment_id          0
order_id            0
payment_datetime    0
method              0
payment_status      0
amount              0
currency            0
dtype: int64


In [45]:
print("--- ANÁLISIS DE DOMINIOS CATEGÓRICOS ---")

#extraer limites para el check 
metodos_unicos = df_payments['method'].unique()
estados_pago = df_payments['payment_status'].unique()
monedas_unicas = df_payments['currency'].unique()

print(f"Métodos de pago autorizados: {metodos_unicos}\n")
print(f"Estados de transacción: {estados_pago}\n")
print(f"Monedas operativas: {monedas_unicas}")

print("\nDiagnóstico:")
print("Obligatorio implementar CHECK IN para blindar estas tres columnas y evitar inyección de estados o métodos no reconocidos por el negocio.")

--- ANÁLISIS DE DOMINIOS CATEGÓRICOS ---
Métodos de pago autorizados: <StringArray>
['transfer', 'card', 'cash', 'wallet']
Length: 4, dtype: str

Estados de transacción: <StringArray>
['rejected', 'approved', 'pending', 'refunded']
Length: 4, dtype: str

Monedas operativas: <StringArray>
['PYG', 'USD']
Length: 2, dtype: str

Diagnóstico:
Obligatorio implementar CHECK IN para blindar estas tres columnas y evitar inyección de estados o métodos no reconocidos por el negocio.


In [46]:
print("--- AUDITORÍA DE INTEGRIDAD FINANCIERA ---")

#buscqueda de transacciones sin valor
pagos_cero = df_payments[df_payments['amount'] <= 0]
print(f"Alerta: Se detectaron {len(pagos_cero)} intentos de pago con monto 0.0 o inferior.")

#fraudes, pagos aprovados sin fondos
pagos_fantasma = df_payments[(df_payments['amount'] <= 0) & (df_payments['payment_status'] == 'approved')]
print(f"Anomalía Crítica: De esos intentos, {len(pagos_fantasma)} pagos sin fondos figuran como 'approved'.")

print("\nConclusión Innegociable:")
print("El esquema SQL debe tener un muro estricto: CHECK (amount > 0). El sistema actual está regalando productos.")

#mostrar evidencia en pantarlla
display(pagos_fantasma[['payment_id', 'order_id', 'method', 'payment_status', 'amount']].head())

--- AUDITORÍA DE INTEGRIDAD FINANCIERA ---
Alerta: Se detectaron 6911 intentos de pago con monto 0.0 o inferior.
Anomalía Crítica: De esos intentos, 5541 pagos sin fondos figuran como 'approved'.

Conclusión Innegociable:
El esquema SQL debe tener un muro estricto: CHECK (amount > 0). El sistema actual está regalando productos.


,payment_id,order_id,method,payment_status,amount
16,17,91612,transfer,approved,0.0
70,71,105287,transfer,approved,0.0
71,72,80654,wallet,approved,0.0
100,101,113491,card,approved,0.0
121,122,14463,transfer,approved,0.0


## Archivo order_items.csv

In [47]:
print("=== PERFILADO DE ORDER_ITEMS.CSV ===")
df_order_items = pd.read_csv('data/order_items.csv')

print("\n1. Tipos de datos por fila:\n")
print(df_order_items.dtypes)

print("\n2. Valores nulos por fila:\n")
print(df_order_items.isnull().sum())

=== PERFILADO DE ORDER_ITEMS.CSV ===

1. Tipos de datos por fila:

order_item_id      int64
order_id           int64
product_id         int64
quantity           int64
unit_price       float64
discount_rate    float64
line_total       float64
dtype: object

2. Valores nulos por fila:

order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
discount_rate    0
line_total       0
dtype: int64


In [49]:
print("--- ANÁLISIS DE INTEGRIDAD RELACIONAL ---")

#buscar si el mismo producto fue registrado 2 veces en el mismo pedido
duplicados_logicos = df_order_items.duplicated(subset=['order_id', 'product_id']).sum()

print(f"Alerta: Se detectaron {duplicados_logicos} combinaciones de order_id + product_id duplicadas.")
print("\nDiagnóstico:")
print("Para evitar que el front-end envíe basura y fragmente los pedidos, es obligatorio levantar un muro combinado en SQL:")
print("UNIQUE (order_id, product_id) -> Esto fuerza a que la cantidad de un mismo producto se consolide en una sola fila.")

--- ANÁLISIS DE INTEGRIDAD RELACIONAL ---
Alerta: Se detectaron 60 combinaciones de order_id + product_id duplicadas.

Diagnóstico:
Para evitar que el front-end envíe basura y fragmente los pedidos, es obligatorio levantar un muro combinado en SQL:
UNIQUE (order_id, product_id) -> Esto fuerza a que la cantidad de un mismo producto se consolide en una sola fila.


In [48]:
print("--- AUDITORÍA MATEMATICA FINANCIERA ---")

#buscar precios irreales o cantidades irreales
errores_fisicos = df_order_items[(df_order_items['quantity'] <= 0) | (df_order_items['unit_price'] <= 0)]
print(f"Cantidades o precios físicos imposibles (<=0): {len(errores_fisicos)} registros.")

#buscar descuentos raros
errores_descuento = df_order_items[(df_order_items['discount_rate'] < 0) | (df_order_items['discount_rate'] > 1)]
print(f"Descuentos ilógicos (fuera del rango 0.0 - 1.0): {len(errores_descuento)} registros.")

print("\nConclusión Innegociable:")
print("El esquema SQL de order_items debe incluir los siguientes CHECKs para que la matematica no colapse:")
print("1. CHECK (quantity > 0)")
print("2. CHECK (unit_price > 0)")
print("3. CHECK (line_total >= 0)")
print("4. CHECK (discount_rate >= 0 AND discount_rate <= 1)")

--- AUDITORÍA MATEMATICA FINANCIERA ---
Cantidades o precios físicos imposibles (<=0): 0 registros.
Descuentos ilógicos (fuera del rango 0.0 - 1.0): 0 registros.

Conclusión Innegociable:
El esquema SQL de order_items debe incluir los siguientes CHECKs para que la matematica no colapse:
1. CHECK (quantity > 0)
2. CHECK (unit_price > 0)
3. CHECK (line_total >= 0)
4. CHECK (discount_rate >= 0 AND discount_rate <= 1)


## Archivo order_status_history.csv

In [50]:
print("=== PERFILADO DE ORDER_STATUS_HISTORY.CSV ===")
df_history = pd.read_csv('data/order_status_history.csv')

print("\n1. Tipos de datos por fila:\n")
print(df_history.dtypes)

print("\n2. Valores nulos por fila:\n")
print(df_history.isnull().sum())

=== PERFILADO DE ORDER_STATUS_HISTORY.CSV ===

1. Tipos de datos por fila:

status_history_id    int64
order_id             int64
status                 str
changed_at             str
changed_by             str
reason                 str
dtype: object

2. Valores nulos por fila:

status_history_id         0
order_id                  0
status                    0
changed_at                0
changed_by                0
reason               225031
dtype: int64


In [51]:
print("--- ANÁLISIS DE DOMINIOS CATEGÓRICOS ---")

#ver valores unicos para limitar la tabla luego
estados_historial = df_history['status'].unique()
actores_autorizados = df_history['changed_by'].unique()

print(f"Estados de pedido registrados: {estados_historial}\n")
print(f"Actores/Sistemas que ejecutan cambios: {actores_autorizados}")

print("\nDiagnóstico:")
print("Para evitar que un bug o una inyección externa registre estados inventados o actores falsos (ej. changed_by = 'hacker'), es estrictamente \nnecesario implementar reglas CHECK IN en ambas columnas.")

--- ANÁLISIS DE DOMINIOS CATEGÓRICOS ---
Estados de pedido registrados: <StringArray>
['shipped', 'packed', 'delivered', 'created', 'paid', 'refunded', 'cancelled']
Length: 7, dtype: str

Actores/Sistemas que ejecutan cambios: <StringArray>
['system', 'user', 'ops', 'warehouse', 'payment_gateway']
Length: 5, dtype: str

Diagnóstico:
Para evitar que un bug o una inyección externa registre estados inventados o actores falsos (ej. changed_by = 'hacker'), es estrictamente 
necesario implementar reglas CHECK IN en ambas columnas.


In [52]:
print("--- AUDITORÍA DE LÓGICA DE NEGOCIO (RAZONES DE CANCELACIÓN) ---")

#aislar los cancelados
cancelaciones = df_history[df_history['status'] == 'cancelled']

#buscar los que no estan justificados
cancelaciones_mudas = cancelaciones[cancelaciones['reason'].isnull()]

print(f"Total de cambios a estado 'cancelled': {len(cancelaciones)}")
print(f"Alerta Operativa: De esas cancelaciones, {len(cancelaciones_mudas)} no tienen un motivo registrado en 'reason'.")

print("\nConclusión:")
print("A nivel de base de datos (SQL), la columna 'reason' debe permanecer permitiendo NULLs para no bloquear las transiciones automáticas del sistema.")
print("Sin embargo, esta evidencia se documenta para exigir el motivo de la cancelacion")

--- AUDITORÍA DE LÓGICA DE NEGOCIO (RAZONES DE CANCELACIÓN) ---
Total de cambios a estado 'cancelled': 17375
Alerta Operativa: De esas cancelaciones, 0 no tienen un motivo registrado en 'reason'.

Conclusión:
A nivel de base de datos (SQL), la columna 'reason' debe permanecer permitiendo NULLs para no bloquear las transiciones automáticas del sistema.
Sin embargo, esta evidencia se documenta para exigir el motivo de la cancelacion


## Archivo order_audit.csv

In [53]:


print("=== PERFILADO DE ORDER_AUDIT.CSV ===")
df_audit = pd.read_csv('data/order_audit.csv')

print("\n1. Tipos de datos por fila:\n")
print(df_audit.dtypes)

print("\n2. Valores nulos por fila:\n")
print(df_audit.isnull().sum())

=== PERFILADO DE ORDER_AUDIT.CSV ===

1. Tipos de datos por fila:

audit_id      int64
order_id      int64
field_name      str
old_value       str
new_value       str
changed_at      str
changed_by      str
dtype: object

2. Valores nulos por fila:

audit_id      0
order_id      0
field_name    0
old_value     0
new_value     0
changed_at    0
changed_by    0
dtype: int64


In [54]:
print("--- ANÁLISIS DE DOMINIOS Y VECTORES DE CAMBIO ---")

campos_auditados = df_audit['field_name'].unique()
actores_auditoria = df_audit['changed_by'].unique()

print(f"Campos bajo vigilancia estricta: {campos_auditados}\n")
print(f"Actores con permisos de modificación: {actores_auditoria}")

print("\nDiagnóstico:")
print("Implementar CHECK IN en 'field_name' y 'changed_by' para evitar inyección de registros de auditoría falsos sobre columnas inexistentes o por actores no autorizados.")

--- ANÁLISIS DE DOMINIOS Y VECTORES DE CAMBIO ---
Campos bajo vigilancia estricta: <StringArray>
['order_total', 'shipping_address', 'customer_phone', 'current_status',
 'notes']
Length: 5, dtype: str

Actores con permisos de modificación: <StringArray>
['system', 'support', 'ops']
Length: 3, dtype: str

Diagnóstico:
Implementar CHECK IN en 'field_name' y 'changed_by' para evitar inyección de registros de auditoría falsos sobre columnas inexistentes o por actores no autorizados.


In [55]:
print("--- AUDITORÍA DE TIPOS POLIMÓRFICOS ---")

#filtrar la auditoria hechas sobre la columna order_total
auditoria_montos = df_audit[df_audit['field_name'] == 'order_total']

print("Muestra de cambios en la columna 'order_total':")
display(auditoria_montos[['audit_id', 'field_name', 'old_value', 'new_value']].head(3))

print("\nConclusión Innegociable:")
print("A pesar de que 'order_total' es numérico en la tabla de pedidos, en la tabla de auditoría los valores están representados como cadenas \nalfanuméricas (posible hashing o casteo a string).")
print("Por lo tanto, 'old_value' y 'new_value' DEBEN ser de tipo TEXT en SQLite. Usar un tipo numérico aquí provocaría un colapso total al intentar registrar el historial.")

--- AUDITORÍA DE TIPOS POLIMÓRFICOS ---
Muestra de cambios en la columna 'order_total':


,audit_id,field_name,old_value,new_value
0,1,order_total,7DD25R,MWBE2F
13,14,order_total,WWRY9M,YLZL83
22,23,order_total,LOXUR9,B0QEAZ



Conclusión Innegociable:
A pesar de que 'order_total' es numérico en la tabla de pedidos, en la tabla de auditoría los valores están representados como cadenas 
alfanuméricas (posible hashing o casteo a string).
Por lo tanto, 'old_value' y 'new_value' DEBEN ser de tipo TEXT en SQLite. Usar un tipo numérico aquí provocaría un colapso total al intentar registrar el historial.


## Analisis Extra


In [57]:
print("=== AUDITORÍA CRÍTICA: RIESGO CAMBIARIO ===")

#la distribucion de las monedas en el nucleo transaccional
distribucion_monedas = df_orders['currency'].value_counts()
print("Distribución de pedidos por moneda operativa:")
print(distribucion_monedas.to_string())

#demostracion del caos contable
total_bruto_falso = df_orders['order_total'].sum()

print("\nSIMULACIÓN DE COLAPSO CONTABLE:")
print(f"Si un analista ejecuta un SUM() directo sobre la columna 'order_total', el sistema reportará: {total_bruto_falso:,.2f}")
print("Diagnóstico: Este número es basura pura. Representa una suma directa de Guaraníes y Dólares sin aplicar una tasa de conversión cambiaria.")

print("\nDICTAMEN ARQUITECTÓNICO PARA LA FASE SQL:")
print("1. El esquema DDL admitirá ambas monedas mediante CHECK (currency IN ('PYG', 'USD')).")
print("2. REGLA INQUEBRANTABLE: Queda estrictamente prohibido realizar agregaciones (SUM, AVG) sobre la columna 'order_total' o 'amount' sin aislar los datos \npreviamente, o sin cruzar los datos con una tabla de tipo de cambio.")

=== AUDITORÍA CRÍTICA: RIESGO CAMBIARIO ===
Distribución de pedidos por moneda operativa:
currency
PYG    108042
USD     11958

SIMULACIÓN DE COLAPSO CONTABLE:
Si un analista ejecuta un SUM() directo sobre la columna 'order_total', el sistema reportará: 239,361,007.57
Diagnóstico: Este número es basura pura. Representa una suma directa de Guaraníes y Dólares sin aplicar una tasa de conversión cambiaria.

DICTAMEN ARQUITECTÓNICO PARA LA FASE SQL:
1. El esquema DDL admitirá ambas monedas mediante CHECK (currency IN ('PYG', 'USD')).
2. REGLA INQUEBRANTABLE: Queda estrictamente prohibido realizar agregaciones (SUM, AVG) sobre la columna 'order_total' o 'amount' sin aislar los datos 
previamente, o sin cruzar los datos con una tabla de tipo de cambio.
